Step 0: Due to issues, I had to investigate what happend in the voxel representation. This part visualizes the corrected data and outputs them as a .html

In [ ]:
import h5py
import numpy as np
import plotly.graph_objects as go

IN_H5 = "/home/maurice/Documents/Master_Data/Periscope_test/Static/4_RPS/Results/dvSave_corrected_interp_wide.h5"
OUT_HTML = "/home/maurice/Documents/Master_Data/Periscope_test/Static/4_RPS/Results/corrected_full_voxel_grid_pos_crop.html"

# ============================
# DATA
# ============================
USE_POLARITY = "pos"   # "pos" or "neg"

TIME_START_US = 0
TIME_WINDOW_US = 250_000

# ============================
# CROP / RESOLUTION
# ============================
X_MIN = 0
X_MAX = 15000

Y_MIN = 250
Y_MAX = 600

VOXEL_SIZE_XY = 2
VOXEL_SIZE_T_US = 10

MAX_POINTS = 120_000

# ============================
# AXIS ASSIGNMENT
# choose from: "x", "y", "time"
# ============================
AXIS_X = "x"
AXIS_Y = "y"
AXIS_Z = "time"

ASPECT_X = 15.0
ASPECT_Y = 1.0
ASPECT_Z = 1.0


def load_events(path):
    with h5py.File(path, "r") as h5:
        ev = h5[USE_POLARITY][:]

        pan_w = int(h5.attrs.get("PAN_W", ev[:, 0].max() + 1))
        pan_h = int(h5.attrs.get("PAN_H", ev[:, 1].max() + 1))

    ev = ev[np.argsort(ev[:, 2])]
    return ev, pan_w, pan_h


def crop_time(ev):
    t = ev[:, 2].astype(np.int64)

    if TIME_START_US is None:
        t0 = int(t.min())
    else:
        t0 = int(TIME_START_US)

    t1 = t0 + TIME_WINDOW_US
    m = (t >= t0) & (t < t1)

    return ev[m], t0, t1


def crop_spatial(ev):
    x = ev[:, 0]
    y = ev[:, 1]

    m = (
        (x >= X_MIN) & (x <= X_MAX) &
        (y >= Y_MIN) & (y <= Y_MAX)
    )

    return ev[m]


def voxelize(ev):
    x = ev[:, 0].astype(np.int64)
    y = ev[:, 1].astype(np.int64)
    t = ev[:, 2].astype(np.int64)

    t0 = t.min()

    xv = (x - X_MIN) // VOXEL_SIZE_XY
    yv = (y - Y_MIN) // VOXEL_SIZE_XY
    tv = (t - t0) // VOXEL_SIZE_T_US

    keys = np.column_stack([xv, yv, tv])

    uniq, counts = np.unique(
        keys,
        axis=0,
        return_counts=True
    )

    x_plot = X_MIN + uniq[:, 0] * VOXEL_SIZE_XY + VOXEL_SIZE_XY / 2
    y_plot = Y_MIN + uniq[:, 1] * VOXEL_SIZE_XY + VOXEL_SIZE_XY / 2
    t_plot = uniq[:, 2] * VOXEL_SIZE_T_US / 1000.0

    return x_plot, y_plot, t_plot, counts


def downsample(x, y, t, counts):
    n = len(x)

    if n <= MAX_POINTS:
        return x, y, t, counts

    idx = np.random.default_rng(123).choice(n, MAX_POINTS, replace=False)

    return x[idx], y[idx], t[idx], counts[idx]


def axis_data(name, x, y, t):
    if name == "x":
        return x
    if name == "y":
        return y
    if name == "time":
        return t
    raise ValueError(f"Unknown axis: {name}")


def axis_range(name):
    if name == "x":
        return [X_MIN, X_MAX]
    if name == "y":
        return [Y_MAX, Y_MIN]
    if name == "time":
        return [0, TIME_WINDOW_US / 1000.0]
    raise ValueError(f"Unknown axis: {name}")


def main():
    ev, pan_w, pan_h = load_events(IN_H5)
    ev, t0, t1 = crop_time(ev)
    ev = crop_spatial(ev)

    print("polarity:", USE_POLARITY)
    print("events in cropped window:", ev.shape[0])
    print("time:", t0, "to", t1)
    print("PAN_W:", pan_w)
    print("PAN_H:", pan_h)
    print("crop:", X_MIN, X_MAX, Y_MIN, Y_MAX)

    if ev.shape[0] == 0:
        raise RuntimeError("No events in selected crop/time window.")

    x, y, tt, counts = voxelize(ev)
    x, y, tt, counts = downsample(x, y, tt, counts)

    plot_x = axis_data(AXIS_X, x, y, tt)
    plot_y = axis_data(AXIS_Y, x, y, tt)
    plot_z = axis_data(AXIS_Z, x, y, tt)

    color = np.log1p(counts)

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=plot_x,
        y=plot_y,
        z=plot_z,
        mode="markers",
        marker=dict(
            size=3,
            color=color,
            colorscale="Turbo",
            opacity=0.8,
            colorbar=dict(title="log events / voxel")
        ),
        text=[
            f"x={xi:.1f}<br>y={yi:.1f}<br>t={ti:.2f} ms<br>count={ci}"
            for xi, yi, ti, ci in zip(x, y, tt, counts)
        ],
        hoverinfo="text"
    ))

    fig.update_layout(
        title=f"Voxel grid | {USE_POLARITY} only | axes: {AXIS_X}, {AXIS_Y}, {AXIS_Z}",
        scene=dict(
            xaxis_title=AXIS_X,
            yaxis_title=AXIS_Y,
            zaxis_title=AXIS_Z,
            xaxis=dict(range=axis_range(AXIS_X)),
            yaxis=dict(range=axis_range(AXIS_Y)),
            zaxis=dict(range=axis_range(AXIS_Z)),
            aspectmode="manual",
            aspectratio=dict(
                x=ASPECT_X,
                y=ASPECT_Y,
                z=ASPECT_Z
            ),
        ),
        width=1400,
        height=900,
    )

    fig.write_html(OUT_HTML)
    print("Saved:", OUT_HTML)


if __name__ == "__main__":
    main()

polarity: pos
events in cropped window: 3173177
time: 0 to 250000
PAN_W: 8400
PAN_H: 800
crop: 0 15000 250 600
Saved: /home/maurice/Documents/Master_Data/Periscope_test/Static/4_RPS/Results/corrected_full_voxel_grid_pos_crop.html


Motion segmentation based on mp4

In [ ]:
import h5py
import cv2
import numpy as np
from tqdm import tqdm


IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_corrected_interp_wide.h5"

OUT_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation.h5"
OUT_MP4 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation.mp4"

WINDOW_US = 5_000

# Number of initial windows used as static reference.
# Use a range where the scene is mostly static.
REFERENCE_FRAMES = 100

# Residual threshold.
# Increase if too noisy, decrease if missing motion.
THRESHOLD = 35

# Remove tiny blobs
MIN_AREA = 20

FPS = 30
OUTPUT_WIDTH = 1600


def normalize_uint8(img):
    img = img.astype(np.float32)
    m = img.max()
    if m <= 0:
        return np.zeros_like(img, dtype=np.uint8)
    return np.clip((img / m) * 255, 0, 255).astype(np.uint8)


def build_edge_frame(pos_win, neg_win, h, w):
    on_img = np.zeros((h, w), dtype=np.uint16)
    off_img = np.zeros((h, w), dtype=np.uint16)

    if len(pos_win):
        xp = pos_win[:, 0]
        yp = pos_win[:, 1]

        valid = (
            (xp >= 0) & (xp < w) &
            (yp >= 0) & (yp < h)
        )

        np.add.at(on_img, (yp[valid], xp[valid]), 1)

    if len(neg_win):
        xn = neg_win[:, 0]
        yn = neg_win[:, 1]

        valid = (
            (xn >= 0) & (xn < w) &
            (yn >= 0) & (yn < h)
        )

        np.add.at(off_img, (yn[valid], xn[valid]), 1)

    edge = np.abs(
        on_img.astype(np.int16) -
        off_img.astype(np.int16)
    )

    return normalize_uint8(edge)


def clean_motion_mask(mask):
    mask = mask.astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)

    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= MIN_AREA:
            cleaned[labels == i] = 255

    return cleaned


def main():
    with h5py.File(IN_H5, "r") as f:
        pos = f["pos"]
        neg = f["neg"]

        h = int(f.attrs["PAN_H"])
        w = int(f.attrs["PAN_W"])

        t_pos = pos[:, 2].astype(np.int64)
        t_neg = neg[:, 2].astype(np.int64)

        t0 = min(t_pos[0], t_neg[0])
        t1 = max(t_pos[-1], t_neg[-1])

        n_frames = int(np.ceil((t1 - t0) / WINDOW_US))

        print("PAN_W:", w)
        print("PAN_H:", h)
        print("Frames:", n_frames)

        scale = OUTPUT_WIDTH / w
        out_h = int(h * scale)

        writer = cv2.VideoWriter(
            OUT_MP4,
            cv2.VideoWriter_fourcc(*"mp4v"),
            FPS,
            (OUTPUT_WIDTH, out_h),
        )

        if not writer.isOpened():
            raise RuntimeError("Could not open MP4 writer.")

        with h5py.File(OUT_H5, "w") as out:
            edge_ds = out.create_dataset(
                "edges",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            residual_ds = out.create_dataset(
                "residual",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            motion_ds = out.create_dataset(
                "motion_mask",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            times_ds = out.create_dataset(
                "t",
                shape=(n_frames,),
                dtype=np.int64,
            )

            reference_acc = np.zeros((h, w), dtype=np.float32)
            reference_count = 0

            p0 = 0
            n0 = 0

            for frame_idx in tqdm(range(n_frames)):
                win_start = t0 + frame_idx * WINDOW_US
                win_end = win_start + WINDOW_US

                p1 = np.searchsorted(t_pos, win_end, side="left")
                n1 = np.searchsorted(t_neg, win_end, side="left")

                pos_win = pos[p0:p1]
                neg_win = neg[n0:n1]

                edge = build_edge_frame(pos_win, neg_win, h, w)

                if frame_idx < REFERENCE_FRAMES:
                    reference_acc += edge.astype(np.float32)
                    reference_count += 1

                    residual = np.zeros_like(edge)
                    motion_mask = np.zeros_like(edge)

                else:
                    reference = reference_acc / max(reference_count, 1)
                    reference_u8 = np.clip(reference, 0, 255).astype(np.uint8)

                    residual = cv2.absdiff(edge, reference_u8)

                    _, motion_mask = cv2.threshold(
                        residual,
                        THRESHOLD,
                        255,
                        cv2.THRESH_BINARY,
                    )

                    motion_mask = clean_motion_mask(motion_mask)

                edge_ds[frame_idx] = edge
                residual_ds[frame_idx] = residual
                motion_ds[frame_idx] = motion_mask
                times_ds[frame_idx] = win_start

                # Visualization
                edge_bgr = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

                overlay = edge_bgr.copy()

                # red overlay = segmented motion
                overlay[motion_mask > 0] = (0, 255, 255)

                vis = cv2.addWeighted(edge_bgr, 0.65, overlay, 0.35, 0)

                cv2.putText(
                    vis,
                    f"Frame {frame_idx+1}/{n_frames}",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    2,
                )

                if frame_idx < REFERENCE_FRAMES:
                    label = "Building static reference"
                else:
                    label = "Motion segmentation"

                cv2.putText(
                    vis,
                    label,
                    (30, 95),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.0,
                    (255, 255, 255),
                    2,
                )

                vis = cv2.resize(
                    vis,
                    (OUTPUT_WIDTH, out_h),
                    interpolation=cv2.INTER_AREA,
                )

                writer.write(vis)

                p0 = p1
                n0 = n1

            out.attrs["PAN_W"] = w
            out.attrs["PAN_H"] = h
            out.attrs["WINDOW_US"] = WINDOW_US
            out.attrs["REFERENCE_FRAMES"] = REFERENCE_FRAMES
            out.attrs["THRESHOLD"] = THRESHOLD
            out.attrs["method"] = "motion = abs(current_edge - static_reference_edge)"

        writer.release()

    print("Saved H5:", OUT_H5)
    print("Saved MP4:", OUT_MP4)


if __name__ == "__main__":
    main()

PAN_W: 8400
PAN_H: 800
Frames: 1405


100%|██████████| 1405/1405 [04:54<00:00,  4.78it/s]

Saved H5: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation.h5
Saved MP4: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation.mp4


A second approach in Motion segmentation with an adaptive mask

In [ ]:
import h5py
import cv2
import numpy as np
from tqdm import tqdm


IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_corrected_interp_wide.h5"

OUT_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation_adaptive.h5"
OUT_MP4 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation_adaptive.mp4"

WINDOW_US = 33_000

REFERENCE_FRAMES = 30

THRESHOLD = 60
MIN_AREA = 50

ALPHA = 0.005

FPS = 30
OUTPUT_WIDTH = 1600


def normalize_uint8(img):
    img = img.astype(np.float32)
    m = img.max()
    if m <= 0:
        return np.zeros_like(img, dtype=np.uint8)
    return np.clip((img / m) * 255, 0, 255).astype(np.uint8)


def build_edge_frame(pos_win, neg_win, h, w):
    on_img = np.zeros((h, w), dtype=np.uint16)
    off_img = np.zeros((h, w), dtype=np.uint16)

    if len(pos_win):
        xp = pos_win[:, 0]
        yp = pos_win[:, 1]
        valid = (xp >= 0) & (xp < w) & (yp >= 0) & (yp < h)
        np.add.at(on_img, (yp[valid], xp[valid]), 1)

    if len(neg_win):
        xn = neg_win[:, 0]
        yn = neg_win[:, 1]
        valid = (xn >= 0) & (xn < w) & (yn >= 0) & (yn < h)
        np.add.at(off_img, (yn[valid], xn[valid]), 1)

    edge = np.abs(on_img.astype(np.int16) - off_img.astype(np.int16))
    return normalize_uint8(edge)


def clean_motion_mask(mask):
    mask = mask.astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)

    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= MIN_AREA:
            cleaned[labels == i] = 255

    return cleaned


def main():
    with h5py.File(IN_H5, "r") as f:
        pos = f["pos"]
        neg = f["neg"]

        h = int(f.attrs["PAN_H"])
        w = int(f.attrs["PAN_W"])

        t_pos = pos[:, 2].astype(np.int64)
        t_neg = neg[:, 2].astype(np.int64)

        t0 = min(t_pos[0], t_neg[0])
        t1 = max(t_pos[-1], t_neg[-1])

        n_frames = int(np.ceil((t1 - t0) / WINDOW_US))

        print("PAN_W:", w)
        print("PAN_H:", h)
        print("Frames:", n_frames)

        scale = OUTPUT_WIDTH / w
        out_h = int(h * scale)

        writer = cv2.VideoWriter(
            OUT_MP4,
            cv2.VideoWriter_fourcc(*"mp4v"),
            FPS,
            (OUTPUT_WIDTH, out_h),
        )

        if not writer.isOpened():
            raise RuntimeError("Could not open MP4 writer.")

        with h5py.File(OUT_H5, "w") as out:
            edge_ds = out.create_dataset(
                "edges",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            residual_ds = out.create_dataset(
                "residual",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            motion_ds = out.create_dataset(
                "motion_mask",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            background_ds = out.create_dataset(
                "background",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            times_ds = out.create_dataset(
                "t",
                shape=(n_frames,),
                dtype=np.int64,
            )

            reference_acc = np.zeros((h, w), dtype=np.float32)
            reference_count = 0
            background = None

            p0 = 0
            n0 = 0

            for frame_idx in tqdm(range(n_frames)):
                win_start = t0 + frame_idx * WINDOW_US
                win_end = win_start + WINDOW_US

                p1 = np.searchsorted(t_pos, win_end, side="left")
                n1 = np.searchsorted(t_neg, win_end, side="left")

                pos_win = pos[p0:p1]
                neg_win = neg[n0:n1]

                edge = build_edge_frame(pos_win, neg_win, h, w)
                edge_f = edge.astype(np.float32)

                if frame_idx < REFERENCE_FRAMES:
                    reference_acc += edge_f
                    reference_count += 1

                    residual = np.zeros_like(edge)
                    motion_mask = np.zeros_like(edge)

                    background_vis = normalize_uint8(reference_acc)

                else:
                    if background is None:
                        background = reference_acc / max(reference_count, 1)

                    residual_f = np.abs(edge_f - background)

                    motion_bool = residual_f > THRESHOLD
                    motion_mask = (motion_bool.astype(np.uint8) * 255)

                    motion_mask = clean_motion_mask(motion_mask)
                    motion_bool_clean = motion_mask > 0

                    # Adaptive background update:
                    # update only where no motion was detected
                    update_region = ~motion_bool_clean

                    background[update_region] = (
                        (1.0 - ALPHA) * background[update_region]
                        + ALPHA * edge_f[update_region]
                    )

                    residual = np.clip(residual_f, 0, 255).astype(np.uint8)
                    background_vis = np.clip(background, 0, 255).astype(np.uint8)

                edge_ds[frame_idx] = edge
                residual_ds[frame_idx] = residual
                motion_ds[frame_idx] = motion_mask
                background_ds[frame_idx] = background_vis
                times_ds[frame_idx] = win_start

                edge_bgr = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

                overlay = edge_bgr.copy()

                # yellow motion overlay, OpenCV BGR
                overlay[motion_mask > 0] = (0, 255, 255)

                vis = cv2.addWeighted(edge_bgr, 0.55, overlay, 0.45, 0)

                cv2.putText(
                    vis,
                    f"Frame {frame_idx+1}/{n_frames}",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    2,
                )

                if frame_idx < REFERENCE_FRAMES:
                    label = "Building adaptive reference"
                else:
                    label = "Adaptive motion segmentation"

                cv2.putText(
                    vis,
                    label,
                    (30, 95),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.0,
                    (255, 255, 255),
                    2,
                )

                vis = cv2.resize(
                    vis,
                    (OUTPUT_WIDTH, out_h),
                    interpolation=cv2.INTER_AREA,
                )

                writer.write(vis)

                p0 = p1
                n0 = n1

            out.attrs["PAN_W"] = w
            out.attrs["PAN_H"] = h
            out.attrs["WINDOW_US"] = WINDOW_US
            out.attrs["REFERENCE_FRAMES"] = REFERENCE_FRAMES
            out.attrs["THRESHOLD"] = THRESHOLD
            out.attrs["MIN_AREA"] = MIN_AREA
            out.attrs["ALPHA"] = ALPHA
            out.attrs["method"] = "adaptive background: motion = abs(edge - background), update background only outside motion"

        writer.release()

    print("Saved H5:", OUT_H5)
    print("Saved MP4:", OUT_MP4)


if __name__ == "__main__":
    main()

PAN_W: 8400
PAN_H: 800
Frames: 213


100%|██████████| 213/213 [01:08<00:00,  3.12it/s]

Saved H5: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation_adaptive.h5
Saved MP4: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_motion_segmentation_adaptive.mp4


Third attempt with enhanced edges

In [ ]:
import h5py
import cv2
import numpy as np
from tqdm import tqdm


IN_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/dvSave_corrected_interp_wide.h5"

OUT_STATIC_MP4 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_static_edges.mp4"
OUT_MOTION_MP4 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_motion_edges.mp4"
OUT_H5 = "/home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_motion_static.h5"

WINDOW_US = 5_000
FPS = 30
OUTPUT_WIDTH = 1600

# edge activity threshold
EDGE_ACTIVE_THRESH = 20

# stability update speed
# lower = slower, more conservative static model
STABILITY_ALPHA = 0.005

# pixels with stability above this are considered static
STATIC_THRESH = 0.45

# pixels with stability below this can become motion
MOTION_STABILITY_MAX = 0.35

# remove small false blobs
MIN_AREA = 50

# optional smoothing to tolerate small static jitter
BLUR_KERNEL = 5


def normalize_uint8(img):
    img = img.astype(np.float32)
    m = img.max()
    if m <= 0:
        return np.zeros_like(img, dtype=np.uint8)
    return np.clip((img / m) * 255, 0, 255).astype(np.uint8)


def clean_mask(mask):
    mask = mask.astype(np.uint8)

    kernel = np.ones((3, 3), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)

    cleaned = np.zeros_like(mask)

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= MIN_AREA:
            cleaned[labels == i] = 255

    return cleaned


def build_edge_frame(pos_win, neg_win, h, w):
    on_img = np.zeros((h, w), dtype=np.uint16)
    off_img = np.zeros((h, w), dtype=np.uint16)

    if len(pos_win):
        xp = pos_win[:, 0]
        yp = pos_win[:, 1]

        valid = (
            (xp >= 0) & (xp < w) &
            (yp >= 0) & (yp < h)
        )

        np.add.at(on_img, (yp[valid], xp[valid]), 1)

    if len(neg_win):
        xn = neg_win[:, 0]
        yn = neg_win[:, 1]

        valid = (
            (xn >= 0) & (xn < w) &
            (yn >= 0) & (yn < h)
        )

        np.add.at(off_img, (yn[valid], xn[valid]), 1)

    edge = np.abs(
        on_img.astype(np.int16) -
        off_img.astype(np.int16)
    )

    edge = normalize_uint8(edge)

    if BLUR_KERNEL and BLUR_KERNEL > 1:
        edge = cv2.GaussianBlur(edge, (BLUR_KERNEL, BLUR_KERNEL), 0)

    return edge


def make_writer(path, w, h):
    writer = cv2.VideoWriter(
        path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        FPS,
        (w, h),
    )

    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer: {path}")

    return writer


def main():
    with h5py.File(IN_H5, "r") as f:
        pos = f["pos"]
        neg = f["neg"]

        h = int(f.attrs["PAN_H"])
        w = int(f.attrs["PAN_W"])

        t_pos = pos[:, 2].astype(np.int64)
        t_neg = neg[:, 2].astype(np.int64)

        t0 = min(t_pos[0], t_neg[0])
        t1 = max(t_pos[-1], t_neg[-1])

        n_frames = int(np.ceil((t1 - t0) / WINDOW_US))

        scale = OUTPUT_WIDTH / w
        out_h = int(h * scale)

        static_writer = make_writer(OUT_STATIC_MP4, OUTPUT_WIDTH, out_h)
        motion_writer = make_writer(OUT_MOTION_MP4, OUTPUT_WIDTH, out_h)

        with h5py.File(OUT_H5, "w") as out:
            edge_ds = out.create_dataset(
                "edges",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            static_ds = out.create_dataset(
                "static",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            motion_ds = out.create_dataset(
                "motion",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            stability_ds = out.create_dataset(
                "stability",
                shape=(n_frames, h, w),
                dtype=np.uint8,
                compression="gzip",
                chunks=(1, h, w),
            )

            times_ds = out.create_dataset(
                "t",
                shape=(n_frames,),
                dtype=np.int64,
            )

            stability = np.zeros((h, w), dtype=np.float32)

            p0 = 0
            n0 = 0

            print("PAN_W:", w)
            print("PAN_H:", h)
            print("Frames:", n_frames)

            for i in tqdm(range(n_frames)):
                win_start = t0 + i * WINDOW_US
                win_end = win_start + WINDOW_US

                p1 = np.searchsorted(t_pos, win_end, side="left")
                n1 = np.searchsorted(t_neg, win_end, side="left")

                pos_win = pos[p0:p1]
                neg_win = neg[n0:n1]

                edge = build_edge_frame(pos_win, neg_win, h, w)

                edge_active = (edge > EDGE_ACTIVE_THRESH).astype(np.float32)

                stability = (
                    (1.0 - STABILITY_ALPHA) * stability
                    + STABILITY_ALPHA * edge_active
                )

                static_mask = stability > STATIC_THRESH

                motion_mask = (
                    (edge > EDGE_ACTIVE_THRESH) &
                    (stability < MOTION_STABILITY_MAX)
                )

                motion_mask = (motion_mask.astype(np.uint8) * 255)
                motion_mask = clean_mask(motion_mask)

                static_img = np.zeros_like(edge)
                static_img[static_mask] = edge[static_mask]

                motion_img = np.zeros_like(edge)
                motion_img[motion_mask > 0] = edge[motion_mask > 0]

                stability_vis = np.clip(stability * 255, 0, 255).astype(np.uint8)

                edge_ds[i] = edge
                static_ds[i] = static_img
                motion_ds[i] = motion_img
                stability_ds[i] = stability_vis
                times_ds[i] = win_start

                static_bgr = cv2.cvtColor(static_img, cv2.COLOR_GRAY2BGR)
                motion_bgr = np.zeros((h, w, 3), dtype=np.uint8)

                # yellow motion
                motion_bgr[motion_mask > 0] = (0, 255, 255)

                cv2.putText(
                    static_bgr,
                    "Static / stable edges",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (255, 255, 255),
                    2,
                )

                cv2.putText(
                    motion_bgr,
                    "Motion / unstable edges",
                    (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0, 255, 255),
                    2,
                )

                static_vis = cv2.resize(
                    static_bgr,
                    (OUTPUT_WIDTH, out_h),
                    interpolation=cv2.INTER_AREA,
                )

                motion_vis = cv2.resize(
                    motion_bgr,
                    (OUTPUT_WIDTH, out_h),
                    interpolation=cv2.INTER_AREA,
                )

                static_writer.write(static_vis)
                motion_writer.write(motion_vis)

                p0 = p1
                n0 = n1

            out.attrs["PAN_W"] = w
            out.attrs["PAN_H"] = h
            out.attrs["WINDOW_US"] = WINDOW_US
            out.attrs["EDGE_ACTIVE_THRESH"] = EDGE_ACTIVE_THRESH
            out.attrs["STABILITY_ALPHA"] = STABILITY_ALPHA
            out.attrs["STATIC_THRESH"] = STATIC_THRESH
            out.attrs["MOTION_STABILITY_MAX"] = MOTION_STABILITY_MAX
            out.attrs["MIN_AREA"] = MIN_AREA
            out.attrs["method"] = "stability map: repeated edge activity becomes static; low-stability active edges become motion"

        static_writer.release()
        motion_writer.release()

    print("Saved static MP4:", OUT_STATIC_MP4)
    print("Saved motion MP4:", OUT_MOTION_MP4)
    print("Saved H5:", OUT_H5)


if __name__ == "__main__":
    main()

PAN_W: 8400
PAN_H: 800
Frames: 1405


100%|██████████| 1405/1405 [05:32<00:00,  4.22it/s]

Saved static MP4: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_static_edges.mp4
Saved motion MP4: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_motion_edges.mp4
Saved H5: /home/maurice/Documents/Master_Data/Periscope/Static/4_RPS/Results/stability_motion_static.h5
